# Stage 02: Image Preprocessing

**Status:** Implemented. Deterministic, dataset-agnostic RGB → Gamma Correction → CLAHE →
Processed RGB PNG -- no other transform. See `PROJECT_CODE.md`'s "Stage 02 Preprocessing Policy"
and `PROJECT_STRUCTURE.md`'s Pipeline Overview for the full specification this notebook
implements; `SEGMENTATION_ARCHITECTURE.md` for why RGB (not single-channel) and no resize.

## Objective

Apply `image_preprocessing.py`'s approved Stage 02 pipeline (Gamma Correction, then CLAHE; no
Green Channel Extraction, Ben Graham, Median Filtering, Histogram Equalization, resizing, or
augmentation -- all explicitly excluded) to every fundus image in the datasets this project has
approved, writing the result once to each dataset's `processed/` folder for every downstream
stage to reuse. This notebook does not train anything -- it is a deterministic transform, not a
trained model, so it has no Model Creation / Training / TensorBoard sections the way
`stage01_iqa.ipynb` does.

## Stage 02 Preprocessing Policy (`PROJECT_CODE.md`)

> Stage 02 preprocessing is deterministic. Each dataset is preprocessed exactly once. Processed
> outputs are stored and reused by every downstream stage. No downstream stage should regenerate
> deterministic preprocessing outputs.

## Datasets processed by this run

- **APTOS2019** -- `raw/train_images` (3,662 images) and `raw/test_images` (1,928 images).
- **IDRiD (grading subset)** -- `raw/1. Original Images/a. Training Set` (413 images) and
  `b. Testing Set` (103 images).

## Datasets intentionally NOT processed here

- **EyeQ** -- excluded by policy. Stage 1 (Image Quality Assessment) is trained on, and must
  continue to see, the original unprocessed RGB EyeQ images (`config.PREPROCESSING_PROFILES.IQA`
  is a no-op specifically for this reason). Stage 02 never touches `datasets/EyeQ/`.
- **DRIVE, CHASE_DB1** -- approved for Stage 03 training, but not yet placed under
  `datasets/DRIVE/raw` / `datasets/CHASE_DB1/raw` (see `IMPLEMENTATION_PLAN.md`'s migration
  notes). Re-run this notebook's job list once they exist -- no code changes needed, only a new
  entry in the `JOBS` list below.
- **IDRiD localization subset** -- its `raw/1. Original Images` is byte-for-byte identical to the
  grading subset's (verified by SHA-256 during the dataset-structure review that preceded this
  implementation) -- the same source photographs, different labels. Reprocessing it would violate
  the "each dataset is preprocessed exactly once" policy above for no benefit; any future consumer
  of localization-labeled images should read from `datasets/IDRiD/grading/processed/` by filename
  instead.
- **IDRiD segmentation subset** -- `datasets/IDRiD/segmentation/raw/` is currently empty (not yet
  populated). Add it to `JOBS` once it exists.

## Before running

`Runtime > Change runtime type > Hardware accelerator > None` -- unlike Stage 1, this stage is a
CPU-bound OpenCV transform (Gamma LUT + CLAHE); it does not use a GPU, so Section 2 below does not
require one (`require_gpu=False`).

### Bootstrap

Identical to `stage01_iqa.ipynb`'s Bootstrap cell -- the minimal clone + `sys.path` setup every
stage notebook needs before `colab/common/` is importable. See `colab/common/setup.py`'s module
docstring for why this is intentionally duplicated rather than imported.

In [ ]:
import os
import subprocess
import sys

REPO_URL = "https://github.com/yasodharan27/diabetic_retinoplasty.git"
REPO_DIR = "/content/diabetic_retinoplasty"
BRANCH = "main"

if not os.path.isdir(os.path.join(REPO_DIR, ".git")):
    subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull", "origin", BRANCH], check=True)

for path in (REPO_DIR, os.path.join(REPO_DIR, "colab", "common")):
    if path not in sys.path:
        sys.path.insert(0, path)

print("Bootstrap complete:", REPO_DIR)

## 1. Setup

`setup.setup()` mounts Google Drive, clones/updates the repository, installs
`requirements.txt`, and enters the repository -- identical call to `stage01_iqa.ipynb`'s Section 1,
reused unmodified. (It also points `EYEQ_RAW_DIR` at Drive; harmless here since this notebook
never reads EyeQ.)

In [ ]:
import setup

setup_info = setup.setup()

## 2. Verification

`verify_environment.verify_all()` checks Python/TensorFlow versions, the repository path, Google
Drive, and required packages -- same call as Stage 1, except **`require_gpu=False`**: Stage 02 is
a CPU-bound OpenCV transform (Gamma correction + CLAHE), not a trained model, so it does not need
a GPU runtime the way Stage 1's EfficientNetB0 training does.

This section then stages APTOS2019 and IDRiD's grading subset from Google Drive onto the Colab
VM's local SSD (`dataset_staging.stage_dataset()`, reused unmodified -- dataset-agnostic by
design), verifies each copy is complete (`dataset_staging.verify_staged_copy()`), and runs the new
`verify_dataset.verify_image_folder()` against each raw split for a corruption spot check (the
generic, dataset-agnostic counterpart to `verify_eyeq_dataset()`, which assumes EyeQ's
`labels.csv` structure that APTOS2019/IDRiD don't have).

In [ ]:
import colab_config
import verify_environment

env_report = verify_environment.verify_all(
    repo_dir=colab_config.REPO_DIR,
    drive_mount_point=colab_config.DRIVE_MOUNT_POINT,
    requirements_path=os.path.join(colab_config.REPO_DIR, "requirements.txt"),
    require_gpu=False,
)

In [ ]:
import shutil

# One-time safeguard against a partial local copy left behind by an interrupted
# previous session -- on a clean VM these have nothing to delete and are a no-op.
for name in ("APTOS2019", "IDRiD_grading"):
    local_path = f"/content/datasets/{name}"
    if os.path.exists(local_path):
        shutil.rmtree(local_path)

print("Deleted any incomplete staged datasets.")

In [ ]:
import posixpath

import dataset_staging

# Each dataset is staged once as a whole (raw/ root), covering every split
# (train/test) in a single copy -- cheaper than one stage_dataset() call per
# split, and dataset_staging.py is already dataset-agnostic, so no changes
# were needed to reuse it here.
staged_aptos2019 = dataset_staging.stage_dataset(
    posixpath.join(colab_config.APTOS2019_DATASET_DIR, "raw"),
    "APTOS2019",
)
dataset_staging.verify_staged_copy(staged_aptos2019)

staged_idrid_grading = dataset_staging.stage_dataset(
    posixpath.join(colab_config.IDRID_DATASET_DIR, "grading", "raw"),
    "IDRiD_grading",
)
dataset_staging.verify_staged_copy(staged_idrid_grading)

In [ ]:
import verify_dataset

RAW_SPLITS = {
    "APTOS2019/train_images": os.path.join(staged_aptos2019.local_dir, "train_images"),
    "APTOS2019/test_images": os.path.join(staged_aptos2019.local_dir, "test_images"),
    "IDRiD_grading/a. Training Set": os.path.join(
        staged_idrid_grading.local_dir, "1. Original Images", "a. Training Set"
    ),
    "IDRiD_grading/b. Testing Set": os.path.join(
        staged_idrid_grading.local_dir, "1. Original Images", "b. Testing Set"
    ),
}

raw_reports = {
    label: verify_dataset.verify_image_folder(path)
    for label, path in RAW_SPLITS.items()
}

## 3. Preprocessing Configuration

`profile="DR"` (`config.PREPROCESSING_PROFILES.DR`) is Stage 02's approved recipe: Gamma
Correction using `config.PREPROCESSING.DEFAULT_GAMMA`, then CLAHE using
`config.PREPROCESSING.DEFAULT_CLAHE_CLIP_LIMIT` / `DEFAULT_CLAHE_TILE_GRID_SIZE` -- nothing here
overrides those centralized defaults, and nothing about this stage is dataset-specific (the same
profile runs identically over APTOS2019 and IDRiD). Printed below for this run's own record, since
this notebook has no `experiment_manager`-style `metadata.json` to log it in (Stage 02 is not a
training run).

In [ ]:
import config
from image_preprocessing import preprocess_folder

PROFILE = "DR"

print(f"profile: {PROFILE}")
print(f"gamma: {config.PREPROCESSING.DEFAULT_GAMMA}")
print(f"clahe_clip_limit: {config.PREPROCESSING.DEFAULT_CLAHE_CLIP_LIMIT}")
print(f"clahe_tile_grid_size: {config.PREPROCESSING.DEFAULT_CLAHE_TILE_GRID_SIZE}")

## 4. Run Preprocessing

Reads and writes entirely on the local SSD (both `input_dir` and `output_dir` below are under
`/content/datasets/`), avoiding the per-file Google Drive FUSE latency `dataset_staging.py`'s own
docstring documents -- Section 6 copies the finished local output back to Drive in bulk afterward,
once, rather than this loop writing one file at a time across the Drive mount. Each job also
writes a `log_file` CSV (already part of `preprocess_folder()`) as the per-image manifest --
image name, status, and processing time -- satisfying this stage's manifest requirement without
new code.

In [ ]:
JOBS = [
    ("APTOS2019/train_images", os.path.join(staged_aptos2019.local_dir, "train_images"),
     "/content/processed/APTOS2019/train_images"),
    ("APTOS2019/test_images", os.path.join(staged_aptos2019.local_dir, "test_images"),
     "/content/processed/APTOS2019/test_images"),
    ("IDRiD_grading/a. Training Set",
     os.path.join(staged_idrid_grading.local_dir, "1. Original Images", "a. Training Set"),
     "/content/processed/IDRiD_grading/a. Training Set"),
    ("IDRiD_grading/b. Testing Set",
     os.path.join(staged_idrid_grading.local_dir, "1. Original Images", "b. Testing Set"),
     "/content/processed/IDRiD_grading/b. Testing Set"),
]

LOG_DIR = "/content/processed/_logs"
os.makedirs(LOG_DIR, exist_ok=True)

preprocessing_results = {}
for label, input_dir, output_dir in JOBS:
    log_file = os.path.join(LOG_DIR, label.replace("/", "_") + ".csv")
    result = preprocess_folder(input_dir, output_dir, profile=PROFILE, log_file=log_file)
    preprocessing_results[label] = result
    print(f"[{label}] processed={result.summary.processed} "
          f"skipped={result.summary.skipped} failed={result.summary.failed}")

## 5. Verify Output

Confirms every job produced zero failures (the pipeline's own correctness gate -- Stage 02 has no
accuracy metric to check, only "did every accepted image actually get processed"), then re-runs
`verify_dataset.verify_image_folder()` against each local processed folder as an independent
check that the written files are themselves valid, decodable images -- not just that
`preprocess_folder()` reported success.

In [ ]:
for label, result in preprocessing_results.items():
    if result.summary.failed > 0:
        print(f"[WARN] {label}: {result.summary.failed} image(s) failed to preprocess -- "
              f"see {os.path.join(LOG_DIR, label.replace('/', '_') + '.csv')} for details.")
    assert result.summary.total_images == (
        result.summary.processed + result.summary.skipped + result.summary.failed
    ), f"{label}: summary counts do not add up"

processed_reports = {
    label: verify_dataset.verify_image_folder(output_dir)
    for label, _, output_dir in JOBS
}

print("\nAll processed output folders verified (see per-folder counts above).")

## 6. Export to Drive

Bulk-copies each job's finished local `processed/` folder (plus its manifest CSV) up to the
corresponding Drive-mounted `datasets/<name>/processed/` destination -- once, after processing is
complete, rather than writing each file across the Drive mount individually during Section 4. The
copy helper below mirrors `dataset_staging.py`'s own `_copy_one` / thread-pool pattern (same
rationale: Drive's FUSE mount is latency-bound per file open, so overlapping many small-file
copies is substantially faster than a sequential copy) for the reverse direction --
`dataset_staging.py` itself only copies Drive-to-local, so this is the smallest amount of new glue
needed to reuse that same proven approach for local-to-Drive, without modifying that module.

In [ ]:
import concurrent.futures
import shutil as _shutil

def _copy_one_to_drive(src, dst):
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    _shutil.copy2(src, dst)

def copy_tree_to_drive(local_dir, drive_dir, max_workers=16):
    file_pairs = []
    for dirpath, _, filenames in os.walk(local_dir):
        rel_dir = os.path.relpath(dirpath, local_dir)
        dest_dir = drive_dir if rel_dir == "." else os.path.join(drive_dir, rel_dir)
        for name in filenames:
            file_pairs.append((os.path.join(dirpath, name), os.path.join(dest_dir, name)))

    with concurrent.futures.ThreadPoolExecutor(max_workers=max_workers) as pool:
        futures = [pool.submit(_copy_one_to_drive, src, dst) for src, dst in file_pairs]
        for future in concurrent.futures.as_completed(futures):
            future.result()
    return len(file_pairs)

DRIVE_DESTINATIONS = {
    "APTOS2019/train_images": posixpath.join(colab_config.APTOS2019_DATASET_DIR, "processed", "train_images"),
    "APTOS2019/test_images": posixpath.join(colab_config.APTOS2019_DATASET_DIR, "processed", "test_images"),
    "IDRiD_grading/a. Training Set": posixpath.join(colab_config.IDRID_DATASET_DIR, "grading", "processed", "a. Training Set"),
    "IDRiD_grading/b. Testing Set": posixpath.join(colab_config.IDRID_DATASET_DIR, "grading", "processed", "b. Testing Set"),
}

exported_counts = {}
for label, _, local_output_dir in JOBS:
    drive_dir = DRIVE_DESTINATIONS[label]
    count = copy_tree_to_drive(local_output_dir, drive_dir)
    log_src = os.path.join(LOG_DIR, label.replace("/", "_") + ".csv")
    log_dst = posixpath.join(posixpath.dirname(drive_dir), "_logs", os.path.basename(log_src))
    _copy_one_to_drive(log_src, log_dst)
    exported_counts[label] = count
    print(f"[{label}] exported {count} file(s) -> {drive_dir}")

## 7. Final Summary

In [ ]:
print("=" * 72)
print("STAGE 02 PREPROCESSING -- RUN SUMMARY")
print("=" * 72)
for label, _, local_output_dir in JOBS:
    result = preprocessing_results[label]
    print(f"{label}:")
    print(f"  raw images:       {raw_reports[label].image_count}")
    print(f"  processed:        {result.summary.processed}  "
          f"skipped: {result.summary.skipped}  failed: {result.summary.failed}")
    print(f"  exported to:      {DRIVE_DESTINATIONS[label]}  ({exported_counts[label]} files)")

print("\nNot processed this run: EyeQ (excluded by policy, Stage 1 only), DRIVE/CHASE_DB1 (not yet")
print("on disk), IDRiD localization (byte-identical to grading -- reuse grading/processed/),")
print("IDRiD segmentation (raw/ empty).")
print("\nNext step in the pipeline (see PROJECT_CODE.md / IMPLEMENTATION_PLAN.md): Stage 03 --")
print("Vessel Segmentation, once DRIVE and CHASE_DB1 are placed under datasets/.")